# Text Mining Lab

In [3]:
import pandas as pd

df = pd.read_csv('/Users/christinelezama/Downloads/amazon_labelled.txt', sep='\t')

print(df.head())

                                                text  sentiment
0  So there is no way for me to plug it in here i...          0
1                        Good case, Excellent value.          1
2                             Great for the jawbone.          1
3  Tied to charger for conversations lasting more...          0
4                                  The mic is great.          1


In [4]:
s = df.loc[26, 'text']

print(s)

I've owned this phone for 7 months now and can say that it's the best mobile phone I've had.


In [7]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

sTokenized = word_tokenize(s)

print(sTokenized)
print(s.split())

['I', "'ve", 'owned', 'this', 'phone', 'for', '7', 'months', 'now', 'and', 'can', 'say', 'that', 'it', "'s", 'the', 'best', 'mobile', 'phone', 'I', "'ve", 'had', '.']
["I've", 'owned', 'this', 'phone', 'for', '7', 'months', 'now', 'and', 'can', 'say', 'that', "it's", 'the', 'best', 'mobile', 'phone', "I've", 'had.']


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/christinelezama/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/christinelezama/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Question 1: 

NLTK splits up contractions and punctuation, which makes the review easier to clean. The split() method only breaks the text at spaces, so words like “I've” stay together.

In [10]:
def fix_contractions(word):
    if word == "'ve":
        return 'have'
    elif word == "'s":
        return 'is'
    else:
        return word

sTokenized = [fix_contractions(word) for word in sTokenized]

print(sTokenized)

['I', 'have', 'owned', 'this', 'phone', 'for', 'months', 'now', 'and', 'can', 'say', 'that', 'it', 'is', 'the', 'best', 'mobile', 'phone', 'I', 'have', 'had']


In [15]:
sTokenized = [word for word in sTokenized if word.isalpha()]

print(sTokenized)

['i', 'have', 'owned', 'this', 'phone', 'for', 'months', 'now', 'and', 'can', 'say', 'that', 'it', 'is', 'the', 'best', 'mobile', 'phone', 'i', 'have', 'had']


In [18]:
sTokenized = [word.lower() for word in sTokenized]

print(sTokenized)

['own', 'phone', 'month', 'say', 'best', 'mobil', 'phone']


In [17]:
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

nltk.download('stopwords')

stemmer = PorterStemmer()
stopWords = stopwords.words('english')

sTokenized = [
    stemmer.stem(word)
    for word in sTokenized
    if word not in stopWords
]

print(len(sTokenized))
print(sTokenized)

7
['own', 'phone', 'month', 'say', 'best', 'mobil', 'phone']


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/christinelezama/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Question 2: 

There are 7 tokens remaining after cleaning the text, removing stop words, and stemming. The final list is own, phone, month, say, best, mobil and phone.

In [24]:
from sklearn.feature_extraction.text import CountVectorizer

analyzer = CountVectorizer().build_analyzer()

def process_text(doc):
    return [
        stemmer.stem(word)
        for word in analyzer(doc)
        if word not in stopWords and word.isalpha()
    ]

In [27]:
from sklearn.model_selection import train_test_split

text_train, text_test, y_train, y_test = train_test_split(
    df['text'],
    df['sentiment'],
    test_size=0.3,
    random_state=42
)

In [34]:
vectorizer = CountVectorizer(analyzer=process_text)

X_train = vectorizer.fit_transform(text_train)
X_test = vectorizer.transform(text_test)

In [38]:
dtm = pd.DataFrame(
    X_train.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print(dtm.head())

   abhor  abil  abl  abound  absolut  absolutel  ac  accept  access  \
0      0     0    0       0        0          0   0       0       0   
1      0     0    0       0        0          0   0       0       0   
2      0     0    0       0        0          0   0       0       0   
3      0     0    0       0        0          0   0       0       0   
4      0     0    0       0        0          0   0       0       0   

   accessori  ...  worthless  worthwhil  would  wow  wrong  ye  year  yell  \
0          0  ...          0          0      0    0      0   0     0     0   
1          0  ...          0          0      0    0      0   0     0     0   
2          0  ...          0          0      0    0      0   0     0     0   
3          0  ...          0          0      0    0      0   0     0     0   
4          0  ...          0          0      0    0      0   0     0     0   

   yet  zero  
0    0     0  
1    0     0  
2    0     0  
3    0     0  
4    0     0  

[5 rows x 110

In [46]:
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

np.random.seed(42)

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_pred)
dt_precision = precision_score(y_test, dt_pred)
dt_recall = recall_score(y_test, dt_pred)

print('DT Accuracy:', dt_accuracy)
print('DT Precision:', dt_precision)
print('DT Recall:', dt_recall)

DT Accuracy: 0.7766666666666666
DT Precision: 0.8041958041958042
DT Recall: 0.7467532467532467


In [53]:
from sklearn.neural_network import MLPClassifier

np.random.seed(42)

nn = MLPClassifier()
nn.fit(X_train, y_train)

nn_pred = nn.predict(X_test)

nn_accuracy = accuracy_score(y_test, nn_pred)
nn_precision = precision_score(y_test, nn_pred)
nn_recall = recall_score(y_test, nn_pred)

print('NN Accuracy:', nn_accuracy)
print('NN Precision:', nn_precision)
print('NN Recall:', nn_recall)

NN Accuracy: 0.8133333333333334
NN Precision: 0.8141025641025641
NN Recall: 0.8246753246753247


Question 3: 

The decision tree had an accuracy of 77.67%, precision of 80.42%, and recall of 74.68%. The neural network had an accuracy of 81.33%, precision of 81.41%, and recall of 82.47%. The neural network performed better on all three measures.

In [58]:
nltk.download('vader_lexicon')

from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/christinelezama/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [63]:
def positive_score(text):
    return sid.polarity_scores(text)['pos']

def negative_score(text):
    return sid.polarity_scores(text)['neg']

df['positive'] = df['text'].apply(positive_score)
df['negative'] = df['text'].apply(negative_score)

print(df[['text', 'positive', 'negative']].head())

                                                text  positive  negative
0  So there is no way for me to plug it in here i...     0.000     0.120
1                        Good case, Excellent value.     0.900     0.000
2                             Great for the jawbone.     0.577     0.000
3  Tied to charger for conversations lasting more...     0.000     0.287
4                                  The mic is great.     0.577     0.000


In [68]:
def predict_sentiment(row):
    if row['positive'] > row['negative']:
        return 1
    else:
        return 0

df['vader_prediction'] = df.apply(predict_sentiment, axis=1)

print(df[['sentiment', 'vader_prediction']].head())

   sentiment  vader_prediction
0          0                 0
1          1                 1
2          1                 1
3          0                 0
4          1                 1


In [72]:
vader_accuracy = accuracy_score(
    df['sentiment'],
    df['vader_prediction']
)

print('VADER Accuracy (%):', vader_accuracy * 100)

VADER Accuracy (%): 83.2


Question 4: 

VADER’s predictions matched the original sentiment labels for 83.2% of the reviews. It had the highest accuracy, followed by the neural network at 81.33% and the decision tree at 77.67%. Based on these results, the dictionary-based method performed best.